# Build & score the policy triage panel

Notebook version of `build_panel.py` — run cells top to bottom and watch the output: cleaned previews, the integrated panel, and the ranked `policy_review_priority_score`.

**Needs:** `build_panel.py` in the same folder, plus `pandas` and `numpy` (`pip install pandas numpy`). Sections after the self-test need the raw data — run `pull_statcan.ipynb` first.

> **Governance:** the score is a **triage signal for human review only** — it does not establish eligibility or need, and the housing field is a labelled **proxy**. See `metadata/integration_notes.md`.

## Load the pipeline

In [ ]:
import pandas as pd
from build_panel import (CONFIG, clean_statcan, read_raw, lfs_features, income_features,
                         low_income_features, population_features, cpi_features,
                         integrate, score, build, selftest, PROCESSED)
pd.set_option('display.max_columns', 40)
print('Loaded. Score weights:', CONFIG['weights'])

## A. Self-test (always works, no network)
Proves the full clean -> integrate -> score pipeline on synthetic data.

In [ ]:
selftest()

## B. Build from your downloaded data
*(Run `pull_statcan.ipynb` first so `data/raw/` is populated.)*

### Load raw tables

In [ ]:
lfs_raw = read_raw('labour_force', 14100287)
inc_raw = read_raw('income', 11100239)
low_raw = read_raw('income', 11100135)
pop_raw = read_raw('demographics', 17100005)
cpi_raw = read_raw('housing_affordability', 18100004)
print('Labour force loaded:', None if lfs_raw is None else lfs_raw.shape)

### Clean + preview (Labour Force Survey)

In [ ]:
assert lfs_raw is not None, 'Run pull_statcan.ipynb first to download the raw data.'
lfs_c = clean_statcan(lfs_raw, 14100287, 'monthly')
print('Cleaned LFS:', lfs_c.shape)
lfs_c[['ref_date','geo','value','missing_value_flag','data_frequency']].head()

### Feature preview (rates, youth, 12-month changes)

In [ ]:
lfs_f = lfs_features(lfs_c)
print('LFS features:', lfs_f.shape)
lfs_f[['geo','ref_date','unemployment_rate','employment_rate','participation_rate',
       'youth_unemployment_rate','unemp_change']].tail()

### Build everything (clean all sources, integrate, score, write `processed/`)

In [ ]:
build()   # writes data/processed/*_clean.csv and policy_triage_panel.csv

## C. The triage result — latest period per province, ranked

In [ ]:
panel = pd.read_csv(PROCESSED / 'policy_triage_panel.csv', low_memory=False)
latest = (panel.sort_values('ref_date').groupby('geo').tail(1)
          .sort_values('policy_review_priority_score', ascending=False))
cols = ['geo','ref_date','unemployment_rate','youth_unemployment_rate','low_income_rate',
        'housing_pressure_proxy','policy_review_priority_score','score_confidence','missing_value_flag']
latest[[c for c in cols if c in latest.columns]]

### Why each top region was flagged (evidence)

In [ ]:
for _, r in latest.head(5).iterrows():
    print(f"{r['geo']} ({r['ref_date']}): score {r['policy_review_priority_score']}, "
          f"confidence {r['score_confidence']}")
    print('   ', r['score_explanation'], '\n')

> The score flags areas of **elevated labour-market stress for human policy review** — it must not be used to approve, deny, or reduce benefits. Any interpretation must go through the AI Council before it is treated as decision-ready.